In [1]:
!nvidia-smi

Thu Mar 12 14:13:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /content
!rm -rf llm-ml-educational-assistant
!git clone https://github.com/Nusultan11/llm-ml-educational-assistant.git
%cd /content/llm-ml-educational-assistant
!git log -1 --oneline

/content
Cloning into 'llm-ml-educational-assistant'...
remote: Enumerating objects: 358, done.
remote: Counting objects: 100% (358/358), done.
remote: Compressing objects: 100% (237/237), done.
Receiving objects: 100% (358/358), 284.83 KiB | 8.14 MiB/s, done.
remote: Total 358 (delta 158), reused 303 (delta 103), pack-reused 0 (from 0)
Resolving deltas: 100% (158/158), done.
/content/llm-ml-educational-assistant
4ceb6ad (HEAD -> main, origin/main, origin/HEAD) Set final colab retrieval profile (700/40/5) and record selection evidence


In [3]:
!pip install -q -r requirements.txt
!pip install -q -e .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llm-ml-assistant (pyproject.toml) ... done


In [9]:
from pathlib import Path
from datetime import datetime, timezone
import shutil
from google.colab import files

In [5]:
root = Path("/content/llm-ml-educational-assistant")
print("run_retrieval_ablation.py:", (root / "scripts" / "run_retrieval_ablation.py").exists())
print("colab_light.yaml:", (root / "configs" / "colab_light.yaml").exists())
print((root / "configs" / "colab_light.yaml").read_text(encoding="utf-8"))

run_retrieval_ablation.py: True
colab_light.yaml: True
﻿project:
  name: "llm-ml-assistant"
  version: "0.1"

model:
  name: "microsoft/Phi-3-mini-4k-instruct"
  device: "cuda"
  max_tokens: 256

embeddings:
  name: "BAAI/bge-base-en"

rag:
  # Colab free-friendly profile: hybrid retrieval + lighter generation model.
  chunk_size: 700
  chunk_overlap: 40
  top_k: 5
  retrieval_mode: "hybrid"
  rrf_k: 60

paths:
  data_dir: "data/examples"
  artifacts_dir: "artifacts"
  logs_dir: "logs"



In [6]:
!python scripts/prepare_datasets.py --out-dir data/processed --max-openassistant 30000 --max-dolly 15000
!python scripts/clean_processed_datasets.py --in-dir data/processed --out-dir data/processed_v2_clean_plus --rag-docs-dir data/rag_docs_v2_clean_plus --min-rag-chars 120 --min-instruction-chars 10 --min-response-chars 80 --max-non-ascii-ratio 0.3


Loading OpenAssistant...
README.md: 10.2kB [00:00, 28.5MB/s]
data/train-00000-of-00001-b42a775f407cee(…): 100% 39.5M/39.5M [00:03<00:00, 11.5MB/s]
data/validation-00000-of-00001-134b8fd0c(…): 100% 2.08M/2.08M [00:01<00:00, 2.08MB/s]
Generating train split: 100% 84437/84437 [00:00<00:00, 171808.54 examples/s]
Generating validation split: 100% 4401/4401 [00:00<00:00, 151833.29 examples/s]
Loading Dolly...
README.md: 8.20kB [00:00, 26.0MB/s]
databricks-dolly-15k.jsonl: 100% 13.1M/13.1M [00:00<00:00, 16.3MB/s]
Generating train split: 100% 15011/15011 [00:00<00:00, 192394.45 examples/s]
Loading local StackOverflow JSONL (if exists)...
Loading local ArXiv JSONL (if exists)...
Done
{
  "rag_rows": 9235,
  "sft_rows": 5095,
  "rag_path": "data/processed/rag_corpus.jsonl",
  "sft_path": "data/processed/sft_instructions.jsonl"
}
{
  "input_dir": "data/processed",
  "output_dir": "data/processed_v2_clean_plus",
  "rag_docs_dir": "data/rag_docs_v2_clean_plus",
  "params": {
    "min_rag_chars": 12

In [7]:
!python scripts/build_eval_from_cleaned.py --sft-path data/processed_v2_clean_plus/sft_instructions.jsonl --out data/processed_v2_clean_plus/eval_auto_qa.json --max-items 200

{
  "sft_path": "data/processed_v2_clean_plus/sft_instructions.jsonl",
  "out_path": "data/processed_v2_clean_plus/eval_auto_qa.json",
  "items": 200,
  "seed": 42
}


In [8]:
!python scripts/run_retrieval_ablation.py \
  --base-config configs/colab_light.yaml \
  --rag-docs-dir data/rag_docs_v2_clean_plus \
  --processed-clean-dir data/processed_v2_clean_plus \
  --eval data/processed_v2_clean_plus/eval_auto_qa.json \
  --chunk-sizes 700 \
  --chunk-overlaps 40 \
  --top-ks 5 \
  --retrieval-modes vector,hybrid \
  --tag-prefix colab_mode


Run id: 20260312T141439Z
Profiles: 2 | Variants: 2

=== Build profile 1/2 [('vector', 700, 40)] ===
/usr/bin/python3 /content/llm-ml-educational-assistant/scripts/run_local_pipeline.py --config reports/retrieval_metrics/ablation/20260312T141439Z/configs/01_colab_mode_vector_c700_o40_k5.yaml --artifacts-dir artifacts --processed-clean-dir data/processed_v2_clean_plus --rag-docs-dir data/rag_docs_v2_clean_plus --skip-prepare --snapshot-label 20260312T141439Z_01_colab_mode_vector_c700_o40_k5 --snapshot-notes 'automated retrieval ablation run; profile=('"'"'vector'"'"', 700, 40); seed_tag=colab_mode_vector_c700_o40_k5' --skip-clean

=== Build index ===
/usr/bin/python3 -m llm_ml_assistant.cli index --config reports/retrieval_metrics/ablation/20260312T141439Z/configs/01_colab_mode_vector_c700_o40_k5.yaml --data-dir data/rag_docs_v2_clean_plus --artifacts-dir artifacts --rebuild
modules.json: 100% 349/349 [00:00<00:00, 1.67MB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 7

In [11]:
root = Path("/content/llm-ml-educational-assistant")
ablation_root = root / "reports" / "retrieval_metrics" / "ablation"

runs = sorted([p for p in ablation_root.glob("*") if p.is_dir()], key=lambda p: p.stat().st_mtime)
if not runs:
    raise RuntimeError(f"No ablation runs found in: {ablation_root}")

src_run = runs[-1]
run_id = src_run.name
print("Using run:", run_id)

bundle = root / "colab_export_mode_run"
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir(parents=True, exist_ok=True)

# run folder
dst_run = bundle / "retrieval_metrics" / "ablation" / run_id
dst_run.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(src_run, dst_run, dirs_exist_ok=True)

# history
hist = root / "reports" / "retrieval_metrics" / "history.jsonl"
if hist.exists():
    (bundle / "retrieval_metrics").mkdir(parents=True, exist_ok=True)
    shutil.copy2(hist, bundle / "retrieval_metrics" / "history.jsonl")

# cleaning summary
clean = root / "data" / "processed_v2_clean_plus" / "cleaning_summary.json"
if clean.exists():
    shutil.copy2(clean, bundle / "cleaning_summary.json")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
zip_path = shutil.make_archive(f"/content/colab_mode_run_{run_id}_{stamp}", "zip", bundle)
print("ZIP:", zip_path)
files.download(zip_path)

Using run: 20260312T141439Z
ZIP: /content/colab_mode_run_20260312T141439Z_20260312T142654Z.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>